In [44]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras

from collections import defaultdict
from tensorflow.keras import layers

In [45]:
tf.__version__, keras.__version__

('2.20.0', '3.13.2')

### Load Data

In [46]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
print(f"{x_train.shape = }, {type(x_train)}")
print(f"{x_test.shape = }, {type(x_test)}")

x_train.shape = (60000, 28, 28), <class 'numpy.ndarray'>
x_test.shape = (10000, 28, 28), <class 'numpy.ndarray'>


### Data Preprocessing

In [47]:
def max_min_values(x_train, x_test):
    print(f"Train\n    Max: {x_train.max()}\n    Min: {x_train.min()}") 
    print(f"Test\n    Max: {x_test.max()}\n    Min: {x_test.min()}")  

max_min_values(x_train, x_test)

Train
    Max: 255
    Min: 0
Test
    Max: 255
    Min: 0


In [48]:
# Normalization (Scaling the range [0, 255] to [0.0, 1.0])
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

max_min_values(x_train, x_test)

Train
    Max: 1.0
    Min: 0.0
Test
    Max: 1.0
    Min: 0.0


In [49]:
# Adding channel dimension (60000, 28, 28) -> (60000, 28, 28, 1) 
print(f"Previous: {x_train.shape = }")
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)
print(f"New: {x_train.shape = }")

Previous: x_train.shape = (60000, 28, 28)
New: x_train.shape = (60000, 28, 28, 1)


In [50]:
n_classes = len(np.unique(y_train))
input_shape = x_train.shape[1:]
n_classes, input_shape

(10, (28, 28, 1))

In [51]:
y_train[:5]

array([5, 0, 4, 1, 9], dtype=uint8)

In [52]:
# convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, n_classes)
y_test = keras.utils.to_categorical(y_test, n_classes)

y_train[:5]

array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]])

### Build Neural Network

In [53]:
class ImageClassifier(keras.Model):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.conv2d_1 = layers.Conv2D(32, (3, 3), activation="relu", name="Conv1")
        self.conv2d_2 = layers.Conv2D(64, (3, 3), activation="relu", name="Conv2")
        self.max_pool_1 = layers.MaxPool2D((2, 2), name="MaxPool1")
        self.max_pool_2 = layers.MaxPool2D((2, 2), name="MaxPool2")
        self.flatten = layers.Flatten(name="Flatten")
        self.dropout = layers.Dropout(0.5, name='Dropout')
        self.outputs = layers.Dense(n_classes, activation="softmax", name="Output")

    def call(self, inputs):
        x = self.conv2d_1(inputs)
        x = self.max_pool_1(x)
        x = self.conv2d_2(x)
        x = self.max_pool_2(x)
        x = self.flatten(x)
        x = self.dropout(x)
        outputs = self.outputs(x)
        return outputs
    
    def build_graph(self):
        x = keras.Input(shape=input_shape, name="Input")
        return keras.Model(inputs=x, outputs=self.call(x), name="ImageClassifier")
    
model = ImageClassifier()
model.build_graph().summary()

Model: "ImageClassifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input (InputLayer)              │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv1 (Conv2D)                  │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPool1 (MaxPooling2D)         │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2 (Conv2D)                  │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPool2 (MaxPooling2D)         │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout (Dropout)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output (Dense)                  │ (None, 10)             │        16,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,826 (136.04 KB)

 Trainable params: 34,826 (136.04 KB)

 Non-trainable params: 0 (0.00 B)

### Train

In [54]:
batch_size = 128
epochs = 15
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
history = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

Epoch 1/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 28s 60ms/step - accuracy: 0.9180 - loss: 0.2934 - val_accuracy: 0.9770 - val_loss: 0.0818
Epoch 2/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 21s 50ms/step - accuracy: 0.9769 - loss: 0.0770 - val_accuracy: 0.9832 - val_loss: 0.0603
Epoch 3/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 38s 42ms/step - accuracy: 0.9823 - loss: 0.0585 - val_accuracy: 0.9865 - val_loss: 0.0480
Epoch 4/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 14s 34ms/step - accuracy: 0.9854 - loss: 0.0467 - val_accuracy: 0.9887 - val_loss: 0.0442
Epoch 5/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 14s 32ms/step - accuracy: 0.9878 - loss: 0.0393 - val_accuracy: 0.9880 - val_loss: 0.0472
Epoch 6/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 14s 34ms/step - accuracy: 0.9893 - loss: 0.0341 - val_accuracy: 0.9887 - val_loss: 0.0445
Epoch 7/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 14s 33ms/step - accuracy: 0.9906 - loss: 0.0302 - val_accuracy: 0.9868 - val_loss: 0.0451
Epoch 8/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 14s 33ms/step - accuracy: 0.9914 - loss: 0.0264 - 

### Evaluate 

In [55]:
score = model.evaluate(x_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Test loss: 0.033193040639162064
Test accuracy: 0.989799976348877


## Gradient Visualization Part

In [66]:
from typing import Any


class GradientMonitor:
    def __init__(self) -> None:
        self.history2 = defaultdict(list)

    def capture(self, gradient, variables):
        for grad, var in zip(gradient, variables):
            if grad is None:
                continue

            grad_norm = tf.norm(grad)
            weight_norm = tf.norm(var)
            grad_weight_ratio = grad_norm / (weight_norm + 1e-12)

            grad_mean = tf.reduce_mean(grad)
            grad_std = tf.math.reduce_std(grad)

            self.history2[var.name].append({
                "grad_norm": grad_norm,
                "weight_norm": weight_norm,
                "grad_weight_ratio": grad_weight_ratio,
                "grad_mean": grad_mean,
                "grad_std": grad_std,
            })


class GradientMonitoredImageClassifier(keras.Model):
    def __init__(self, monitor, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.monitor = monitor

        self.conv2d_1 = layers.Conv2D(32, (3, 3), activation="relu", name="Conv1")
        self.conv2d_2 = layers.Conv2D(64, (3, 3), activation="relu", name="Conv2")
        self.max_pool_1 = layers.MaxPool2D((2, 2), name="MaxPool1")
        self.max_pool_2 = layers.MaxPool2D((2, 2), name="MaxPool2")
        self.flatten = layers.Flatten(name="Flatten")
        self.dropout = layers.Dropout(0.5, name='Dropout')
        self.outputs = layers.Dense(n_classes, activation="softmax", name="Output")

    def call(self, inputs, training=False):
        x = self.conv2d_1(inputs)
        x = self.max_pool_1(x)
        x = self.conv2d_2(x)
        x = self.max_pool_2(x)
        x = self.flatten(x)
        x = self.dropout(x, training=training)
        outputs = self.outputs(x)
        return outputs
    
    def train_step(self, data):
        x, y = data

        with tf.GradientTape() as tape:
            y_pred = self(x, training=True)
            loss = self.compiled_loss(y, y_pred)
        
        gradients = tape.gradient(loss, self.trainable_variables)

        # capture gradient of the train step before updating
        if (
            self.monitor is not None and isinstance(self.monitor, GradientMonitor)
        ):
            self.monitor.capture(gradients, self.trainable_variables)
        
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))

        self.compiled_metrics.update_state(y, y_pred)

        return {m.name: m.result() for m in self.metrics}
    
    def build_graph(self):
        x = keras.Input(shape=input_shape, name="Input")
        return keras.Model(inputs=x, outputs=self.call(x), name="ImageClassifier")
    
model2 = GradientMonitoredImageClassifier(GradientMonitor())
model2.build_graph().summary()

Model: "ImageClassifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input (InputLayer)              │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv1 (Conv2D)                  │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPool1 (MaxPooling2D)         │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2 (Conv2D)                  │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPool2 (MaxPooling2D)         │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout (Dropout)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output (Dense)                  │ (None, 10)             │        16,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,826 (136.04 KB)

 Trainable params: 34,826 (136.04 KB)

 Non-trainable params: 0 (0.00 B)

In [67]:
model2.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy']
)

history2 = model2.fit(
    x_train, y_train, batch_size=batch_size, 
    epochs=2, validation_split=0.1
)

Epoch 1/2


c:\Users\rajba\miniconda3\envs\uwl\Lib\site-packages\keras\src\backend\tensorflow\trainer.py:695: UserWarning: `model.compiled_loss()` is deprecated. Instead, use `model.compute_loss(x, y, y_pred, sample_weight, training)`.
  warnings.warn(
c:\Users\rajba\miniconda3\envs\uwl\Lib\site-packages\keras\src\backend\tensorflow\trainer.py:670: UserWarning: `model.compiled_metrics()` is deprecated. Instead, use e.g.:
```
for metric in self.metrics:
    metric.update_state(y, y_pred)
```

  return self._compiled_metrics_update_state(


422/422 ━━━━━━━━━━━━━━━━━━━━ 17s 36ms/step - accuracy: 0.8904 - loss: 0.1000 - val_accuracy: 0.9768 - val_loss: 0.0818
Epoch 2/2
422/422 ━━━━━━━━━━━━━━━━━━━━ 15s 35ms/step - accuracy: 0.9673 - loss: 0.1000 - val_accuracy: 0.9848 - val_loss: 0.0550


In [70]:
model2.monitor.history2

defaultdict(list,
            {'kernel': [{'grad_norm': <tf.Tensor 'norm/Squeeze:0' shape=() dtype=float32>,
               'weight_norm': <tf.Tensor 'norm_1/Squeeze:0' shape=() dtype=float32>,
               'grad_weight_ratio': <tf.Tensor 'truediv:0' shape=() dtype=float32>,
               'grad_mean': <tf.Tensor 'Mean:0' shape=() dtype=float32>,
               'grad_std': <tf.Tensor 'reduce_std/Sqrt:0' shape=() dtype=float32>},
              {'grad_norm': <tf.Tensor 'norm_4/Squeeze:0' shape=() dtype=float32>,
               'weight_norm': <tf.Tensor 'norm_5/Squeeze:0' shape=() dtype=float32>,
               'grad_weight_ratio': <tf.Tensor 'truediv_2:0' shape=() dtype=float32>,
               'grad_mean': <tf.Tensor 'Mean_2:0' shape=() dtype=float32>,
               'grad_std': <tf.Tensor 'reduce_std_2/Sqrt:0' shape=() dtype=float32>},
              {'grad_norm': <tf.Tensor 'norm_8/Squeeze:0' shape=() dtype=float32>,
               'weight_norm': <tf.Tensor 'norm_9/Squeeze:0' shape=(

In [65]:
history2.history

{'accuracy': [0.8850555419921875, 0.9661852121353149],
 'loss': [0.09999983757734299, 0.09999983757734299],
 'val_accuracy': [0.9785000085830688, 0.9808333516120911],
 'val_loss': [0.0865483209490776, 0.07046619802713394]}